 ## Question or problem definition.

In this notebook, we will search and try basic information on a given dataset. The purpose of this time is not to raise the score, but to confirm the basic flow.

The problem definition of "House Prices: Advanced Regression Techniques" is to predict the final price of each home with 79 explanatory variables which explain the housing of Ames in Iowa from various angles.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib
import matplotlib.pyplot as plt
from scipy.stats import skew
from sklearn.preprocessing import StandardScaler
from scipy import stats
import warnings
warnings.filterwarnings('ignore')
%matplotlib　inline

## Acquire training and testing data.

We start by acquiring the training and testing datasets into Pandas DataFrames. We also combine these datasets to run certain operations on both datasets together.

In [ ]:
df_train=pd.read_csv('../input/train.csv')
df_test=pd.read_csv('../input/test.csv')
combine = [df_train, df_test]

## Analyze by describing data.

Let's see which features are categories or numbers in both data.

In [ ]:
# check the columns
df_train.columns

We can also confirm that the dataset was properly read with read_csv.

Let's visualize the contents of the dataset partly, then check the appearance of the data.

In [ ]:
# preview the data
df_train.head()

In [ ]:
df_test.head()

We can confirm feature of categories or numerical values that are included in the dataset.

### Information on purpose variable 'SalePrice'

Let us calculate the distribution of the numerical value of 'SalePrice'.

In [ ]:
df_train['SalePrice'].describe()

###  Missing values

We will investigate missing values in the dataset. It can be seen that 19 pieces (about 25%) of the features of learning data contain missing values.

In [ ]:
total = df_train.isnull().sum().sort_values(ascending=False)
percent = (df_train.isnull().sum()/df_train.isnull().count()).sort_values(ascending=False)*100
missing_data = pd.concat([total, percent], axis=1, keys=['Total', 'Percent'])
missing_data.head(20)

Further narrowing down, we will look for numerical type features with missing values in each of the train data and test data. First, we will search from the train data.

In [ ]:
#  percentage of missing values for objective variables of non-object type
train_obj=df_train.ix[:, df_train.dtypes != np.object]
total = train_obj.isnull().sum().sort_values(ascending=False)
percent = (train_obj.isnull().sum()/train_obj.isnull().count()).sort_values(ascending=False)*100
missing_data1 = pd.concat([total, percent], axis=1, keys=['Total', 'Percent'])
missing_data1.head()

Next we will search the test data.

In [ ]:
#  percentage of missing values for objective variables of non-object type
test_obj=df_test.ix[:, df_test.dtypes != np.object]
total = test_obj.isnull().sum().sort_values(ascending=False)
percent = (test_obj.isnull().sum()/test_obj.isnull().count()).sort_values(ascending=False)*100
missing_data2 = pd.concat([total, percent], axis=1, keys=['Total', 'Percent'])
missing_data2.head(12)

From the above it turned out that there are as many as 8 of the feature data that the test data has a missing value than the train data.

We should delete 'LotFrontage' and 'GarageYrBlt' because the percentage of missing values is large. 
We will fill in 'MasVnrArea' with average value.
We will also fill in features with other missing values in the test dataset as average value

In [ ]:
df_train = df_train.drop((['LotFrontage','GarageYrBlt']), axis=1)
df_test = df_test.drop((['LotFrontage','GarageYrBlt']), axis=1) 

In [ ]:
df_train['MasVnrArea']= df_train['MasVnrArea'].fillna(df_train['MasVnrArea'].mean())

In [ ]:
df_test['MasVnrArea']= df_test['MasVnrArea'].fillna(df_test['MasVnrArea'].mean())
df_test['BsmtHalfBath']= df_test['BsmtHalfBath'].fillna(df_test['BsmtHalfBath'].mean())
df_test['BsmtFullBath']= df_test['BsmtFullBath'].fillna(df_test['BsmtFullBath'].mean())
df_test['GarageArea']= df_test['GarageArea'].fillna(df_test['GarageArea'].mean())
df_test['BsmtFinSF1']= df_test['BsmtFinSF1'].fillna(df_test['BsmtFinSF1'].mean())
df_test['BsmtFinSF2']= df_test['BsmtFinSF2'].fillna(df_test['BsmtFinSF2'].mean())
df_test['BsmtUnfSF']= df_test['BsmtUnfSF'].fillna(df_test['BsmtUnfSF'].mean())
df_test['TotalBsmtSF']= df_test['TotalBsmtSF'].fillna(df_test['TotalBsmtSF'].mean())
df_test['GarageCars']= df_test['GarageCars'].fillna(df_test['GarageCars'].mean())

### Visualize with a heatmap

Let's visualize the correlation of feature quantities with a heatmap and confirm.

In [ ]:
#correlation matrix
df=df_train.drop(['Id'],axis=1).copy()
corrmat = df.corr()
f, ax = plt.subplots(figsize=(12, 9))
sns.heatmap(corrmat, vmax=.8, square=True);

We can see that 'OverallQual', 'GrLivArea', 'GarageCars',' GarageArea ',' TotalBsmtSF etc are strongly correlated with 'SalePrice'.

## Analyze, identify patterns, and explore the data.

### Analyze by pivoting features
We will explore the relationship with 'SalePrice' about the feature quantity of the object type which does not include the missing value.
From the name of feature quantity we pick up what seems to be strongly related to 'SalePrice' and analyze the mutual relationship with pivot.

### Creating new feature extracting from existing

We can convert the categorical titles to ordinal.

In [ ]:
df_train[['HouseStyle', 'SalePrice']].groupby(['HouseStyle'], as_index=False).mean().sort_values(by='SalePrice', ascending=False)

In [ ]:
title_mapping={"2.5Fin":8,"2Story":7,"1Story":6,"SLvl":5,"2.5Unf":4,"1.5Fin":3,"SFoyer":2,"1.5Unf":1}
for dataset in [df_train,df_test]:
    dataset['HouseStyle']=dataset['HouseStyle'].map(title_mapping)

We will also process 'Heating QC' in the same way.

In [ ]:
df_train[['HeatingQC', 'SalePrice']].groupby(['HeatingQC'], as_index=False).mean().sort_values(by='SalePrice', ascending=False)


In [ ]:
title_mapping={"Ex":5,"Gd":4,"TA":3,"Fa":2,"Po":1}
for dataset in [df_train,df_test]:
    dataset['HeatingQC']=dataset['HeatingQC'].map(title_mapping)

We can complete the quantification of the feature of the object type. We are now ready to delete all feature of the object type.

In [ ]:
df_train = df_train.loc[:, df_train.dtypes != 'object']
df_test = df_test.loc[:, df_test.dtypes != 'object']

### 'SalePrice' correlation matrix (zoomed heatmap style)

Select 10 features including 'SalePrice' which has strong correlation with 'SalePrice' and display it with a heat map.

In [ ]:
#saleprice correlation matrix
k = 10  #number of variables for heatmap
corrmat = df_train.corr()
cols = corrmat.nlargest(k, 'SalePrice')['SalePrice'].index
cm = np.corrcoef(df_train[cols].values.T)
sns.set(font_scale=1.25)
f,ax=plt.subplots(figsize=(12,9))
hm = sns.heatmap(cm, cbar=True, annot=True, square=True, fmt='.2f', annot_kws={'size': 12}, yticklabels=cols.values, xticklabels=cols.values)
plt.show()

- 'GarageArea' and 'GarageCars' are indicators showing the same thing at different angles, so we can see that the correlation coefficient is close.
- Unfortunately, it was confirmed that 'HouseStyle' and 'HeatingQC' are weakly correlated with 'SalePrice'.

### Scatter plots between 'SalePrice' and correlated variables

In order to check for outliers, convert the heatmap to a scatter plot with the pairplot method.

In [ ]:
#scatter plots
sns.set()
sns.pairplot(df_train[cols], size = 3)
plt.show();

### Bivariate analysis

When checking the correlation between 'SalePrice' and 'GrLivArea' with a scatter plots, outlier 2 is found at the lower right. Let's delete this.

In [ ]:
#　bivariate analysis saleprice/grlivarea
var = 'GrLivArea'
data = pd.concat([df_train['SalePrice'], df_train[var]], axis=1)
data.plot.scatter(x=var, y='SalePrice', ylim=(0,800000));

 #### Delete outlier value of scatter plots

In [ ]:
df_train.sort_values(by = 'GrLivArea', ascending = False)[:2]
df_train = df_train.drop(df_train[df_train['Id'] == 1299].index)
df_train = df_train.drop(df_train[df_train['Id'] == 524].index)

Let's check the result of the operation with the scatter plot of 'SalePrice' and 'GrLivArea'.

In [ ]:
#　bivariate analysis saleprice/grlivarea
var = 'GrLivArea'
data = pd.concat([df_train['SalePrice'], df_train[var]], axis=1)
data.plot.scatter(x=var, y='SalePrice', ylim=(0,800000));

In this way the outliers have been properly deleted.

We will look at the heatmap of 'SalePrice' and feature amount best ten 10 again. The correlation coefficient also rises somewhat, which shows that the ranking has changed.

In [ ]:
k = 10  
corrmat = df_train.corr()
cols = corrmat.nlargest(k, 'SalePrice')['SalePrice'].index
cm = np.corrcoef(df_train[cols].values.T)
sns.set(font_scale=1.25)
f,ax=plt.subplots(figsize=(12,9))
hm = sns.heatmap(cm, cbar=True, annot=True, square=True, fmt='.2f', annot_kws={'size': 12}, yticklabels=cols.values, xticklabels=cols.values)
plt.show()

 ## Model, predict and solve the problem.
 
Now we are ready to train the model and predict the solution you need.With more than 60 predictive modeling algorithms, we choose polynomial regression and do training.Of course there are many more accurate algorithms than polynomial regression. The reason is that the algorithm I am a beginner knows is polynomial regression.
We will select multiple explanatory variables to measure the precision of the generalization ability of the model by selecting five variables 'OverallQual', 'GrLivArea', 'TotalBsmtSF', 'GarageCars', '1stFlrSF' and performing multiple regression analysis.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from scipy.stats import zscore
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.model_selection import train_test_split
from sklearn.metrics import  mean_squared_error

X=pd.DataFrame(df_train,columns=df_train.columns)
y = pd.DataFrame(df_train['SalePrice'])

X=X.loc[:,['OverallQual', 'GrLivArea','TotalBsmtSF','GarageCars','1stFlrSF']]

 ### Calculate Root Mean Square Error (RMSE)

In [ ]:
y= y.values
y_pred = X.values
RMSE = np.sqrt(np.mean((y-y_pred)**2))
RMSE

### Compare the decision coefficients with polynomial regression

Coefficient of determination :In statistics, the coefficient of determination, denoted R2 or r2 and pronounced "R squared", is the proportion of the variance in the dependent variable that is predictable from the independent variable(s).[wikipedia]
https://en.wikipedia.org/wiki/Coefficient_of_determination

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=0)

from sklearn.preprocessing import PolynomialFeatures
degree_2 = PolynomialFeatures(degree=2)
degree_3 = PolynomialFeatures(degree=3)
degree_4 = PolynomialFeatures(degree=4)

x_train_2 = degree_2.fit_transform(X_train)
x_train_3 = degree_3.fit_transform(X_train)
x_train_4 = degree_4.fit_transform(X_train)

lin_2d = LinearRegression()
lin_3d = LinearRegression()
lin_4d = LinearRegression()

lin_2d.fit(x_train_2,y_train)
lin_3d.fit(x_train_3,y_train)
lin_4d.fit(x_train_4,y_train)

x_test_2 = degree_2.fit_transform(X_test)
x_test_3 = degree_3.fit_transform(X_test)
x_test_4 = degree_4.fit_transform(X_test)

score_2d = lin_2d.score(x_test_2,y_test)
score_3d = lin_3d.score(x_test_3,y_test)
score_4d = lin_4d.score(x_test_4,y_test)

print("The coefficient of determination in the quadratic equation is %.2f"%(score_2d))
print("The coefficient of determination in the cubic equation is %.2f"%(score_3d))
print("The coefficient of determination in the quartic equation is %.2f"%(score_4d))

From the value of the decision coefficient we will adopt a quadratic equation model.

## Visualize, report, and present the problem solving steps and final solution. Supply or submit the results

### Put the test dataset in the model and predict 'SalePrice'

In [ ]:
X=pd.DataFrame(df_test,columns=df_test.columns)

X_2=X.copy()
X=X.loc[:,['OverallQual', 'GrLivArea','TotalBsmtSF','GarageCars','1stFlrSF']]

from sklearn.preprocessing import PolynomialFeatures
degree_2 = PolynomialFeatures(degree=2)
degree_3 = PolynomialFeatures(degree=3)
degree_4 = PolynomialFeatures(degree=4)

x_test_2 = degree_2.fit_transform(X)
x_test_3 = degree_3.fit_transform(X)
x_test_4 = degree_4.fit_transform(X)

y_pred=lin_2d.predict(x_test_2)
y_pred

In [ ]:
submission = pd.DataFrame(lin_2d.predict(x_test_2))
submission.columns=['SalePrice']
submission=pd.concat([X_2['Id'],submission['SalePrice']],axis=1)
#submission.to_csv('../input/submission.csv', index=False)

#### <b>Finally</b>
- If the feature is not a numerical type and the proportion occupied by the object type can not be ignored, we may be able to find features with strong correlation from it, so further ingenuity is necessary.
- Objective Variance The accuracy of the predicted value has been improved when outliers with feature quantities with strong correlation are deleted. It will be a hint to further improve accuracy.
- If you have enough resources, you may try changing the regression model to challenge.